# Autograd - Demonstration

## Introduction

After doing many different machine learning projects, I decided I wanted a deeper understanding of the maths and algorithms behind libraries such as PyTorch, starting from the ground up. autograd.main is a library of functions and classes making up all the building blocks for a classification model (see the MNIST demo below) built using **only** NumPy. Each component is demonstrated briefly below; however, if you want to take a closer look, go to the main file, where the components are all well documented and the code is fairly readable.

Towards the end of this notebook, a full gradient descent loop is shown before a model already trained on the MNIST dataset is loaded and trained further, again exclusively using this library.

**Full disclosure**:
As someone who taught themselves how to code, I have used AI tools such as Gemini and ChatGPT to explain errors, help with syntax, and spot bugs I may have missed. However, I made sure not to blindly use their code, and **no** function was ever designed or fully created by anyone or anything other than myself.

## Library components

In [ ]:
import numpy as np
import main

In [ ]:
# These test inputs will be used throughout this notebook.
test_inputs = [
    np.array([-1, -0.5, 0, 1, 63.4]),
    [[0, 1, 2], [2, 0.7, -1]],
    (-2, -0.32, 0, 1, 0),
    "This is a string."
]

test_input_targets = [
    np.array([-3, -0.1, 0, 1, 1]),
    [[0, 1, 2], [3, 0.6, -1.1]],
    (2, -0.2, 5.6, 3, 2),
    "This is not a string."
]

In [ ]:
# ReLU
for t in test_inputs:
    try:
        print(f"Input - {t} - Output - {main.ReLU(t)}")
    except:
        print(f"Input - {t} - cannot be passed through ReLU.")

In [ ]:
# backwards_ReLU (the derivative of ReLU)
for t in test_inputs:
    try:
        print(f"Input - {t} - Output - {main.backwards_ReLU(t)}")
    except:
        print(f"Input - {t} - cannot be passed through backwards_ReLU.")

In [ ]:
# Softmax
for t in test_inputs:
    try:
        print(f"Input - {t} - Output - {main.Softmax(t)}")
    except:
        print(f"Input - {t} - cannot be passed through Softmax.")

In [ ]:
# CrossEntropyLoss
for i in range(len(test_inputs)):
    try:
        print(f"Input - {test_inputs[i]} - Target - {main.Softmax(test_input_targets[i])} - Output - {main.CrossEntropyLoss(main.Softmax(test_inputs[i]), main.Softmax(test_input_targets[i]))}")
    except:
        print(f"Input - {test_inputs[i]} - and Target - {test_input_targets[i]} - cannot be passed through CrossEntropyLoss.")

In [ ]:
# Linear layer
test_layer = main.LinearLayer(input_size=3, output_size=1, precision='float32', random_seed=1)

print(f"Layer parameters - {test_layer.parameters}")
print()

for t in test_inputs:
    try:
        print(f"Input - {t} - Output - {test_layer.forward(t)}")
    except:
        print(f"Input - {t} - cannot be passed through the linear layer.")

# Most inputs fail because they have the wrong size for the model

## Demos

### Linear regression

In [ ]:
x_axis_for_data = np.linspace(0, 10, 100)
inputs = np.asarray(np.split(x_axis_for_data, 100))
targets = inputs * 2.65 + 1.6

Simple = main.LinearLayer(input_size=1, output_size=1, precision='float32', random_seed=1)

mse_over_time = []
time = np.linspace(1, 10, 10)

print(f"Initial layer parameters - {Simple.parameters}.")

for i in range(10):
    x = Simple.forward(inputs)
    error = x - targets
    mse_over_time.append(np.mean(error ** 2.0))

    print(f"Epoch {i+1} - Mean error {-np.mean(error)}")

    # Gradient of loss with respect to output for MSE
    error_grad = (2.0 / inputs.shape[0]) * error

    Simple.backwards(error_grad)
    Simple.update_parameters(0.01)

print(f"Final layer parameters - {Simple.parameters}.")

In [ ]:
from matplotlib import pyplot as plt

model_preds = Simple.forward(inputs)

fig, axs = plt.subplots(2)

axs[0].scatter(x=time, y=mse_over_time, color='red', label='Mean Squared Error')
axs[0].plot(time, mse_over_time, color='orange', linestyle='--')

axs[0].set_title("Training Loss Over Time")
axs[0].set_xlabel("Epoch")
axs[0].set_ylabel("Mean Squared Error")
axs[0].grid(True)
axs[0].legend()


axs[1].scatter(x=x_axis_for_data, y=model_preds, color='red', label='Simple model predictions')
axs[1].plot(x_axis_for_data, targets.flatten(), color='orange', linestyle='--', label='Actual training data')

axs[1].set_title("Model predictions vs data")
axs[1].grid(True)
axs[1].legend()


plt.show()

### **MNIST** classification through gradient descent
In this section, a sample of the MNIST dataset is used to show how a model can be trained using this library. After a few iterations, to demonstrate the functionality of this library, a pretrained model (trained on a larger dataset using only this library) will be loaded to show more accurate classification while saving time and computation.

#### Setting up the data

In [ ]:
from pandas import read_csv
from pathlib import Path

data_dir = Path("reduced_MNIST_data")

data_frame = read_csv(data_dir)
data_frame = data_frame.drop(columns=["Unnamed: 0"], errors="ignore")

data = data_frame.to_numpy()
m, n = data.shape

data_dev = data[0:200]
Y_dev = data_dev[:, 0].astype(int)
X_dev = data_dev[:, 1:].astype(np.float32) / 255.0

data_train = data[200:m]
Y_train = data_train[:, 0].astype(int)
X_train = data_train[:, 1:].astype(np.float32) / 255.0

print(X_train.shape)
print(Y_train.shape)
m_train, n_features = X_train.shape

def one_hot(Y):
    Y = np.asarray(Y, dtype=int)
    one_hot_Y = np.zeros((Y.size, Y.max() + 1), dtype=np.float32)
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y

Y_train = one_hot(Y_train)
Y_dev = one_hot(Y_dev)

print(X_train.shape)
print(Y_train.shape)

#### Creating and training a model

In [ ]:
model = main.Model(input_size=784, output_size=10, hidden_size=64,
              number_of_layers=8, activation_function=main.ReLU, normalisation_function=main.Softmax,
              random_seed=11, initialisation_function="He")

In [ ]:
for i in range(100):
    output = model.forward(X_train)
    loss = main.CrossEntropyLoss(output, Y_train)
    model.backwards(output, Y_train, main.CrossEntropyLoss)
    model.update_parameters(5e-2)
    if (i+1) % 10 == 0 or i == 0:
        print(f"Epoch: {i+1}    Loss: {round(loss, 5)}    Accuracy: {round(np.exp(-loss) * 100, 3)}%")
        dev_output = model.forward(X_dev)
        dev_loss = main.CrossEntropyLoss(dev_output, Y_dev)
        print(f"Validation loss: {round(dev_loss, 5)}    Accuracy: {round(np.exp(-dev_loss) * 100, 3)}%")

# Running this cell again will continue training the existing model to 200 epochs.


#### Showcasing the data from the model

In [ ]:
print("Model parameters for Layer 3:")
print(model.parameters["Layer 3"])

print("Model gradients for Layer 3:")
print(model.gradients["Layer 3"])

#### Loading a pretrained model

In [ ]:
from pickle import load
saved_model = main.Model(input_size=784, output_size=10, hidden_size=64,
              number_of_layers=8, activation_function=main.ReLU, normalisation_function=main.Softmax,
              random_seed=11, initialisation_function="He")

parameters_path = Path("pretrained_model_parameters.pkl")

with parameters_path.open("rb") as file:
    loaded_parameters = load(file)

saved_model.set_parameters(loaded_parameters)
print("Retrieved saved model")

In [ ]:
# Displaying the saved model's loss and accuracy
# The training loss is found after all training ended and hard coded
print(f"Saved model's Training loss  : 0.05114    Training Accuracy  : 95.014% ")
output = saved_model.forward(X_dev)
loss = main.CrossEntropyLoss(output, Y_dev)
print(f"Saved model's Validation loss: {round(loss, 5)}    Validation Accuracy: {round(np.exp(-loss) * 100, 3)}% ")

#### Visual demonstration

In [ ]:
def show_prediction(index : int):
    current_image = X_dev[index:index + 1, None]
    probs = model.forward(current_image)
    prediction = np.argmax(probs)
    label = np.argmax(Y_dev[index])
    print(f"Prediction: {prediction}    Label: {label}")

    current_image = current_image.reshape((28, 28)) * 255
    plt.gray()
    plt.imshow(current_image, interpolation='nearest')
    plt.show()

for _ in range(4):
    index = np.random.randint(0, 199)
    show_prediction(index)

## Final thoughts

This library is not very sophisticated or expansive; however, the initial task of creating a system of interconnected functions has been completed, and so it would take only a few minutes to add more components such as:
- A wider variety of activation and normalisation functions
- Averaged gradients over time, reducing the chance of getting stuck at a local minimum
- More initialisation functions for the LinearLayers
- etc.

Regardless, the aim of this project was to understand the maths behind gradient descent, not to create the best library in existence, and that aim has been achieved.

## Acknowledgements and Resources

A huge thanks to the following creators for their resources and explanations:

- 3Blue1Brown for all his videos, particularly the first four in his "Neural Networks" series.
    - https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi

- CodeEmporium for his "Backpropagation by hand" video.
    - https://www.youtube.com/watch?v=12-HUfbyGso&list=WL&index=4&t=1022s

- Samson Zhang for his tutorial "Building a neural network FROM SCRATCH," which first introduced me to ML.
    - https://www.youtube.com/watch?v=w8yWXqWQYmU


As someone who taught themselves how to code, the following were very useful both in this project and others:

- GeeksforGeeks.org for its explanations and examples for Python and common Python libraries.
    - https://www.geeksforgeeks.org/

- As mentioned above, AI tools such as Gemini and ChatGPT were used to explain errors, help with syntax, and spot bugs I may have missed. However, I made sure not to blindly use their code, and **no** function was ever designed or fully created by anyone or anything other than myself.